In [1]:
# === STEP 1: Import Libraries ===
import pandas as pd
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime

In [2]:
# === STEP 2: Connect to the Database ===
conn = sqlite3.connect("stackoverflow.db")

In [3]:
# # === STEP 3: Load and Preview 'posts' Table ===
# posts_df = pd.read_sql_query("SELECT * FROM posts", conn)

In [4]:
# === STEP 3: Load Only Required Columns from 'posts' Table ===
query = """
    SELECT Id, CreationDate
    FROM posts
    WHERE CreationDate IS NOT NULL
"""
posts_df = pd.read_sql_query(query, conn)

print(f"Loaded rows: {len(posts_df)}")
print(posts_df.head())

In [ ]:
print(f"Number of rows in 'posts': {len(posts_df)}")
print(posts_df.head())
print(posts_df.dtypes)

In [ ]:
# === STEP 4: Clean Data ===
posts_df["CreationDate"] = pd.to_datetime(posts_df["CreationDate"], errors="coerce")
posts_df = posts_df.dropna(subset=["CreationDate"])
posts_df = posts_df[posts_df["CreationDate"] <= pd.Timestamp(datetime.today())]
posts_df = posts_df.drop_duplicates(subset=["Id"])

In [ ]:
# === STEP 5: Extract Year-Month from CreationDate ===
posts_df["YearMonth"] = posts_df["CreationDate"].dt.to_period("M").astype(str)

In [ ]:
# === STEP 6: Group by Year-Month and Count Posts ===
monthly_posts = posts_df.groupby("YearMonth").size().reset_index(name="PostCount")

In [ ]:
# === STEP 7: Create Bar Plot with Grouped Year Labels ===
plt.figure(figsize=(12, 6))
ax = sns.barplot(x="YearMonth", y="PostCount", data=monthly_posts, palette="viridis")

plt.title("Number of Posts per Year-Month")
plt.xlabel("Year-Month")
plt.ylabel("Post Count")
plt.xticks(rotation=90, ha='right')

# Grouped labels like in previous charts
year_month_pattern = []
unique_years = monthly_posts["YearMonth"].str.split("-").str[0].unique()

for year in unique_years:
    year_month_pattern.append(year)
    year_month_pattern.extend([' '] * (monthly_posts["YearMonth"].str.startswith(year).sum() - 1))

ax.set_xticklabels(year_month_pattern)

plt.tight_layout()
plt.show()